# Agenda

#1 - few additional notes on log reg

#2 - linear regression and logistic regression are probability based models

#3 - predict_proba for classification

#4 - threshold for classification

#5 - regularization in logistic regression

#6 - coefficients in logistic regression

#7 - advantages and disadvantages of logistic regression ; use cases

# 1) notes on log reg

Picnic problem

Can we go to picnic tomorrow ? 

It's a "yes" or "no" question. 

Amount of sunshine is a major factor

Instead of drawing a straight line like a linear regression model, logistic regression uses a special S-shaped curve. This curve keeps the answer nicely tucked between 0 and 1, just like a probability. 

On one side of the curve, if the sunshine is low, the probability is near 0 (no picnic).

On the other side, if the sunshine is high, the probability is near 1 (yes, picnic!). 

So, logistic regression is really just a way to turn a "yes" or "no" question into a "how likely?" question, based on all the information you have.

Sigmoid function or logistic function - It's a mathematical function that takes any number, from very, very big to very, very small, and squishes it into a number that is always between 0 and 1.

The final number the sigmoid function gives you is the probability for your picnic. A probability above 0.5 might mean "go for it!" while a probability below 0.5 might mean "better bring an umbrella".

It's using its sigmoid function to predict a probability, which is a continuous value between 0 and 1 - the reason why it's called regression.

# 2) probability based linear models

### Linear regression


The Problem: The bank wants to predict how much a customer will spend on their credit card each month.

The bank looks at historical data about each customer, such as:

Their income, Their age and The number of years they've been a customer.

The Probability: When the bank uses a linear regression model, it finds a straight line that best fits all the past data points. This line is its prediction. But the bank also knows its prediction won't be perfect for every single customer. It's predicting an outcome with an inherent amount of random error, or probability. 

The Prediction: The bank might predict, "This customer will spend $500 next month."

The Probabilistic Understanding: The model also understands that the actual amount will probably be close to $500. The full probabilistic prediction is more like, "This customer will spend around $500 next month, with a certain amount of uncertainty." For instance, it might say there's a 95% chance the customer will spend between $450 and $550.


### Logistic regression

The Problem: The bank wants to predict whether a customer will stay or leave (churn) in the next three months. This is a "yes" or "no" question. 

The bank feeds the same kind of data into a logistic regression model, such as:

How long the customer has been with the bank, How often they use the online banking app, and How many complaints they have made. 

The Probability: The model uses the S-shaped (sigmoid) curve to translate this information into a single number: the probability of the customer leaving. 

The Prediction: The model outputs a probability like 0.85. This means there's an 85% chance the customer will leave.

The Classification: The bank decides on a rule: if the probability is over 50%, they classify the customer as a "high risk to churn". In this case, the customer is flagged.

The Probabilistic Understanding: The bank doesn't just get a simple "leave" or "stay" answer. It gets a level of confidence, which helps them prioritize. A customer with a 95% churn probability is more urgent than a customer with a 51% probability, even though both are classified as "high risk". 

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

# ----------------------------
# 1) Load dataset
# ----------------------------
data = fetch_california_housing(as_frame=True)
X = data.data            # 8 numeric predictors
y = data.target          # MedianHouseValue (in 100k USD)

# Optional: quick peek
# print(X.head(), y.head())

# ----------------------------
# 2) Train / Test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------
# 3) Fit Linear Regression
# ----------------------------
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit() # Ordinary least squares algo

conf_intervals = model.conf_int(alpha=0.05) # 95 % confidence interval 
conf_intervals.columns = ['lower_bound', 'upper_bound']

print("Confidence Intervals for Model Coefficients (95%):\n")
print(conf_intervals)

## During prediction - X_test is a batch prediction

new_data_dict = {
    'const': 1,
    'MedInc': 3.5,
    'HouseAge': 30.0,
    'AveRooms': 5.0,
    'AveBedrms': 1.0,
    'Population': 1000.0,
    'AveOccup': 3.0,
    'Latitude': 34.0,
    'Longitude': -118.0,
}

# Convert the dictionary to a DataFrame
new_X = pd.DataFrame([new_data_dict])

predictions = model.get_prediction(new_X)
summary_frame = predictions.summary_frame(alpha=0.05)

print("\nPredictions and Intervals for new data (95% confidence):\n")
print(summary_frame)



Confidence Intervals for Model Coefficients (95%):

            lower_bound  upper_bound
const        -38.233405   -35.650435
MedInc         0.428467     0.444919
HouseAge       0.008561     0.010311
AveRooms      -0.118858    -0.095786
AveBedrms      0.589919     0.700212
Population    -0.000013     0.000005
AveOccup      -0.004742    -0.002831
Latitude      -0.435421    -0.407208
Longitude     -0.449279    -0.419749

Predictions and Intervals for new data (95% confidence):

       mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
0  1.910633  0.00721         1.8965       1.924766       0.49096      3.330307


The summary_frame will give you a table with the following columns:

mean: The predicted value for the median house value (in $100k USD).

obs_ci_lower and mean_ci_upper: Prediction Interval for a Single Observation

The number our model generates — $191,000 — is our best estimate for the median house value in this specific location, based on all the factors we've fed into our analysis, like the area's demographics, age of the homes, etc.

The other two numbers, $49,000 and $333,000, represent our confidence range. Think of it as a 'margin of error' for a single house. We're 95% confident that the final, actual price for a specific house with these characteristics will land somewhere between those two values.

This range helps us manage our expectations. While $191,000 is our most probable outcome, the wide interval tells us there's a lot of natural variability in house prices in this area.

It's a much more realistic forecast than just a single number, and it prepares us for potential market fluctuations.

# 3) predict_proba

predict_proba() returns the probability estimates for each class instead of just the predicted label.

For every sample in X_test, you get a vector of probabilities that sum to 1.

### Example (binary classification):

y_proba[0]  → [0.25, 0.75]

Means:

25% chance it belongs to class 0

75% chance it belongs to class 1

### Example (multiclass, qualities 3–8):

y_proba[0]  → [0.05, 0.10, 0.55, 0.20, 0.05, 0.05]

Highest probability is 0.55 for quality 5 → model predicts class 5.

predict() only gives the most likely class (hard decision).

predict_proba() shows how confident the model is.

Example:

Wine A: [0.99, 0.01] → very confident

Wine B: [0.51, 0.49] → model is unsure

Many metrics (ROC-AUC, Precision-Recall AUC, Log-loss) need probabilities, not just hard labels.

roc_auc_score(y_test, y_proba[:, 1])

### How kNN's predict_proba works

In kNN, the "probability" is simply the proportion of the nearest neighbors that belong to a certain class.

Example: For a new data point, if you set k=5 and 3 of the nearest neighbors belong to class A and 2 belong to class B, the predict_proba method will return a probability of 60% for class A and 40% for class B.

Simple ratio, not a true probability: This is a simple count-based approximation, not a true probability derived from an underlying statistical model. There is no uncertainty or statistical model associated with this "inferred class label".

Coarse estimates: For low values of k, the estimates can be very coarse. For example, with k=3, the only possible non-zero probabilities are 33.3%, 66.7%, or 100%. 

# 4) threshold

### Default Behavior (Threshold = 0.5)

In binary classification, predict() uses a cutoff (threshold) of 0.5.

If the predicted probability for class 1 (P(class=1)) is ≥ 0.5, the model predicts 1; otherwise 0.

P(good wine) = 0.6 → predict good (1)

P(good wine) = 0.4 → predict not_good (0)

In wine quality, “good wine” is much less frequent than “not good”.

##### With imbalance:

Model often predicts the majority class (not_good).

Minority class (good wine) gets missed (false negatives).

Example:
Actual = good wine

Predicted probability = 0.45

Default threshold 0.5 → predicts not_good ❌ (false negative)

### Changing the Threshold

We can lower the threshold from 0.5 → 0.3.

Now:
If P(good wine) ≥ 0.3, predict good wine.

This catches more actual “good wines” (reduces false negatives).

Example with threshold = 0.3:
Actual = good wine

Predicted probability = 0.45

Now predicts good (1) ✔️

### Trade-off

Lowering threshold = higher recall, lower false negatives

But it may increase false positives (predicting good when it’s not good).


### For Wine Example

If the business goal is to not miss good wines, you lower threshold (e.g., 0.3).

If the goal is to only label wines as good when very sure, keep threshold high (e.g., 0.7).

# 5) regularization in logistic reg

### Ridge Logistic Regression
log_ridge = LogisticRegression(penalty="l2", C=0.5, solver="lbfgs")

Best when all features matter, but you want to reduce overfitting.

solver=“lbfgs” - Optimization algorithm: Limited-memory

Efficient, handles L2 penalty well.

Works for binary and multiclass problems.

### Lasso Logistic Regression
log_lasso = LogisticRegression(penalty="l1", C=0.5, solver="saga")

Best when you want feature selection (ignores irrelevant features).

saga is needed because only saga (and liblinear for binary) supports L1 penalty.

Scalable for large datasets.

### ElasticNet Logistic Regression
log_elastic = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=0.7,
    C=0.5
)

Best choice when you have many correlated features and want sparsity but also stability.

Combines L1 + L2 regularization.

##### l1_ratio=0.7
	
Controls the balance between L1 and L2:

l1_ratio=1 → pure Lasso

l1_ratio=0 → pure Ridge

0.7 means 70% L1 (sparsity) + 30% L2 (stability).

### C value

C controls how strict the regularization is (smaller C = stronger penalty).

C=0.5 means moderate regularization (stronger than default C=1.0).

C = 0.1 and L1 (Lasso) regularization => It means out of 100 features, it can drop 40 features
C = 0.4 and L1 (Lasso) - It means out of 100 features, it can drop 20 features

# 6) coefficients of logistic regression

In [9]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

# 1. Load data
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = (iris.target == 2).astype(int)  # Binary classification for 'virginica'

# 2. Fit the logistic regression model
model = LogisticRegression(solver='liblinear', random_state=0).fit(X, y)

# 3. Extract and print the coefficients
print("Coefficients from scikit-learn:\n")
print(model.coef_)
print("\nIntercept from scikit-learn:\n")
print(model.intercept_)

# Combine coefficients with feature names for clarity
feature_names = X.columns
coefficients = pd.DataFrame({'feature': feature_names, 'coefficient': model.coef_[0]})
print("\nCoefficients with feature names:\n")
print(coefficients)

Coefficients from scikit-learn:

[[-1.70751526 -1.53427768  2.47096755  2.55537041]]

Intercept from scikit-learn:

[-1.21470917]

Coefficients with feature names:

             feature  coefficient
0  sepal length (cm)    -1.707515
1   sepal width (cm)    -1.534278
2  petal length (cm)     2.470968
3   petal width (cm)     2.555370


# 7) advantages and disadvantages of logistic regression

### advantages

Simple and efficient: It's a relatively simple algorithm that is easy to implement and computationally efficient, making it a fast and accessible baseline model for many classification tasks.

Highly interpretable: The coefficients of the model can be interpreted as the influence of each predictor on the outcome. This transparency is valuable for explaining how a particular decision or prediction was made.

Outputs probabilities: It provides well-calibrated probabilities for the predicted classes, not just the final classification result. This allows for more nuanced decision-making and prioritizing based on risk.

Robust against overfitting with regularization: While it can overfit on high-dimensional data, regularization techniques (L1 and L2) can be easily applied to prevent this.

Works well on linearly separable data: If the data can be separated by a linear decision boundary, logistic regression performs very efficiently.

### disadvantages

Assumes linearity: The main limitation is the assumption that the independent variables are linearly related to the dependent variable. It can't capture complex or non-linear relationships without manual feature engineering (like PolyNomial features)

Sensitive to outliers: Similar to linear regression, logistic regression can be sensitive to outliers, which can skew the model's coefficients and lead to incorrect predictions.

Requires relevant features: The model's predictive power can degrade if you include irrelevant or redundant features, so proper feature selection is important.

Requires a large sample size: For accurate and stable coefficient estimates, logistic regression generally performs better with a larger sample size.

Can be outperformed by more complex algorithms: While a great baseline, more powerful and flexible algorithms like Neural Networks or ensemble methods can often achieve better performance on complex, non-linear problems.


### use cases

Credit scoring: Financial institutions use logistic regression to predict the probability of a customer defaulting on a loan or credit card based on factors like credit history and income.

Fraud detection: It can be used to identify fraudulent transactions by analyzing patterns in transaction amount, location, and other user data.

Spam detection: Email providers use logistic regression to classify emails as either "spam" or "not spam" based on characteristics like the sender, subject line, and content.

Disease prediction: In healthcare, it helps predict the likelihood of a patient developing a certain disease based on risk factors, and this can be extended to survival prediction.

Customer churn prediction: Businesses can predict whether a customer will cancel their subscription or stop using a service based on usage patterns, customer service interactions, and demographic data.

Marketing campaign response: Marketers can predict whether a customer will respond to a specific campaign (e.g., clicking on an ad or making a purchase) based on their past behavior.

Quality control in manufacturing: It can be used to estimate the probability of part failure in machinery and inform maintenance schedules.

### use cases of linear regression

Sales forecasting: Predicting future sales based on past performance, advertising spending, or seasonal trends.

Real estate prediction: Estimating house prices based on factors like square footage, location, and number of bedrooms.

Risk assessment: In insurance, estimating claims costs based on policyholder information or assessing risk in finance by predicting stock prices.

Trend analysis: Forecasting future trends in economic indicators, market performance, or environmental data.

Customer behavior analysis: Predicting customer spending based on age, income, and past purchasing behavior.